# 🎵 Módulo 1 — Análisis del Mercado Musical
**Fuente:** Last.fm API | **Objetivo:** Entender el mercado + predecir hits

---
## Índice
1. [Imports y configuración](#1)
2. [Recolección de datos](#2)
3. [Limpieza](#3)
4. [EDA — Exploración visual](#4)
5. [ML — Predicción de hits](#5)
6. [Función de predicción final](#6)

---
## 1. Imports y configuración <a id='1'></a>

**Sobre el .env:**  
La API key no debe subirse a GitHub. La forma correcta es guardarla en un archivo `.env`:
```
# Archivo .env (en la raíz del proyecto, al mismo nivel que este notebook)
LASTFM_API_KEY=tu_clave_aqui
```
Y cargarlo así: `API_KEY = os.getenv('LASTFM_API_KEY')`  
Mientras desarrollas localmente, puedes usar la clave directamente (pero nunca la subas a GitHub).

In [ ]:
import os
import time
import ast
import warnings
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# --- Configuración API ---
# Opción 1 (recomendada): carga desde .env
# API_KEY = os.getenv('LASTFM_API_KEY')
# Opción 2: directamente (solo para desarrollo local, NO subir a GitHub)
API_KEY  = '63e059c3c912a3f642daf2372484d183'
BASE_URL = 'http://ws.audioscrobbler.com/2.0/'
DELAY    = 0.25  # segundos entre peticiones (4 req/s, límite Last.fm es 5)

print('✅ Todo listo')
print(f'   API Key: {"✅ cargada" if API_KEY else "❌ falta API_KEY"}')

---
## 2. Recolección de datos <a id='2'></a>

Usamos **tres endpoints** de Last.fm para construir el dataset:

| Endpoint | Qué devuelve |
|---|---|
| `chart.getTopTracks` | Top tracks globales con playcount y listeners |
| `geo.getTopTracks` | Top tracks por país (country garantizado, sin NaN) |
| `tag.getTopTracks` | Top tracks por género/tag |

**Nota importante:** `chart.getTopTracks` y `geo.getTopTracks` **no devuelven el género**.  
Para obtenerlo hay que llamar a `track.getTopTags` por separado (ver sección 2.4).

**Rate limit:** Last.fm permite ~5 peticiones/segundo. Con `DELAY=0.25` hacemos 4/seg → margen de seguridad.

**Si algo falla:** el CSV actúa como checkpoint. Si el proceso se interrumpe, al volver a ejecutar carga lo que ya tenías.

In [ ]:
# --- Función base para llamar a la API ---
# Cualquier endpoint de Last.fm pasa por aquí.
# Si el servidor falla, reintenta con espera creciente (backoff exponencial).

def llamar_api(params, reintentos=3):
    """
    Hace una petición a Last.fm y devuelve el JSON.
    Si hay error de servidor (5xx o 429), reintenta hasta 3 veces.
    Devuelve None si falla definitivamente.
    """
    params['api_key'] = API_KEY
    params['format']  = 'json'

    for intento in range(reintentos):
        try:
            respuesta = requests.get(BASE_URL, params=params, timeout=10)

            # 429 = rate limit, 5xx = error del servidor → reintentar
            if respuesta.status_code in [429, 500, 502, 503, 504]:
                espera = 2 ** intento  # 1s, 2s, 4s...
                print(f'  HTTP {respuesta.status_code} → espero {espera}s (intento {intento+1}/{reintentos})')
                time.sleep(espera)
                continue

            respuesta.raise_for_status()
            return respuesta.json()

        except requests.exceptions.RequestException as e:
            espera = 2 ** intento
            print(f'  Error de red: {e} → espero {espera}s')
            time.sleep(espera)

    print('  ❌ Falló definitivamente')
    return None

print('✅ Función lista')

In [ ]:
# --- 2.1 Top tracks globales (chart.getTopTracks) ---
# Cada página devuelve hasta 200 tracks.
# 150 páginas × 200 = ~30.000 tracks globales.

def recoger_top_tracks_global(n_paginas=150, limite=200):
    tracks = []
    
    for pagina in range(1, n_paginas + 1):
        datos = llamar_api({
            'method': 'chart.getTopTracks',
            'page'  : pagina,
            'limit' : limite
        })
        
        if not datos:
            break
        
        lista = datos.get('tracks', {}).get('track', [])
        if not lista:
            break
        
        for i, t in enumerate(lista):
            # El artista viene como diccionario anidado: {'name': 'BTS', 'mbid': ...}
            artista = t.get('artist', {})
            nombre_artista = artista.get('name', '') if isinstance(artista, dict) else str(artista)
            
            tracks.append({
                'name'       : t.get('name', ''),
                'artist'     : nombre_artista,
                'playcount'  : int(t.get('playcount', 0)),
                'listeners'  : int(t.get('listeners', 0)),
                'duration'   : int(t.get('duration', 0)),  # en segundos
                'mbid'       : t.get('mbid', ''),
                'country'    : 'GLOBAL',
                'genre_tag'  : 'UNKNOWN',  # se rellena después con track.getTopTags
                'rank_global': (pagina - 1) * limite + i + 1,
            })
        
        time.sleep(DELAY)
        
        if pagina % 25 == 0:
            print(f'  Página {pagina}/{n_paginas} | tracks: {len(tracks):,}')
    
    print(f'✅ chart.getTopTracks: {len(tracks):,} tracks')
    return tracks

print('✅ Función lista')

In [ ]:
# --- 2.2 Top tracks por país (geo.getTopTracks) ---
# Ventaja clave: el campo 'country' viene siempre asignado → sin NaN.
# El parámetro country va en minúsculas: 'spain', 'france', etc.

def recoger_top_tracks_por_pais(paises, paginas_por_pais=10, limite=200):
    tracks = []
    
    for pais in paises:
        n_antes = len(tracks)
        
        for pagina in range(1, paginas_por_pais + 1):
            datos = llamar_api({
                'method' : 'geo.getTopTracks',
                'country': pais,
                'page'   : pagina,
                'limit'  : limite
            })
            
            if not datos:
                break
            
            lista = datos.get('tracks', {}).get('track', [])
            if not lista:
                break
            
            for i, t in enumerate(lista):
                artista = t.get('artist', {})
                nombre_artista = artista.get('name', '') if isinstance(artista, dict) else str(artista)
                
                tracks.append({
                    'name'          : t.get('name', ''),
                    'artist'        : nombre_artista,
                    'playcount'     : int(t.get('listeners', 0)),  # geo devuelve 'listeners', no 'playcount'
                    'listeners'     : int(t.get('listeners', 0)),
                    'duration'      : int(t.get('duration', 0)),
                    'mbid'          : t.get('mbid', ''),
                    'country'       : pais,
                    'genre_tag'     : 'UNKNOWN',
                    'rank_global'   : None,
                    'rank_by_country': (pagina - 1) * limite + i + 1,
                })
            
            time.sleep(DELAY)
        
        print(f'  geo.getTopTracks [{pais}]: {len(tracks) - n_antes:,} tracks')
    
    print(f'✅ Total geo: {len(tracks):,} tracks')
    return tracks

print('✅ Función lista')

In [ ]:
# --- 2.3 Top tracks por género/tag (tag.getTopTracks) ---
# Aquí el genre_tag sí viene asignado directamente.

def recoger_top_tracks_por_genero(tags, paginas_por_tag=10, limite=200):
    tracks = []
    
    for tag in tags:
        n_antes = len(tracks)
        
        for pagina in range(1, paginas_por_tag + 1):
            datos = llamar_api({
                'method': 'tag.getTopTracks',
                'tag'   : tag,
                'page'  : pagina,
                'limit' : limite
            })
            
            if not datos:
                break
            
            lista = datos.get('tracks', {}).get('track', [])
            if not lista:
                break
            
            for t in lista:
                artista = t.get('artist', {})
                nombre_artista = artista.get('name', '') if isinstance(artista, dict) else str(artista)
                
                tracks.append({
                    'name'     : t.get('name', ''),
                    'artist'   : nombre_artista,
                    'playcount': int(t.get('playcount', 0)),
                    'listeners': int(t.get('listeners', 0)),
                    'duration' : int(t.get('duration', 0)),
                    'mbid'     : t.get('mbid', ''),
                    'country'  : 'UNKNOWN',
                    'genre_tag': tag,  # ✅ aquí el género ya viene asignado
                    'rank_global'   : None,
                    'rank_by_country': None,
                })
            
            time.sleep(DELAY)
        
        print(f'  tag.getTopTracks [{tag}]: {len(tracks) - n_antes:,} tracks')
    
    print(f'✅ Total por género: {len(tracks):,} tracks')
    return tracks

print('✅ Función lista')

In [ ]:
# --- 2.4 Obtener el género de un track (track.getTopTags) ---
# chart.getTopTracks y geo.getTopTracks NO devuelven el género.
# Para enriquecer esos tracks con genre_tag, llamamos a track.getTopTags.
#
# Los tags de Last.fm son etiquetas de la comunidad.
# Filtramos los que no son géneros reales ('seen live', '2020s', etc.)

TAGS_A_IGNORAR = {
    'seen live', 'favorites', 'favourite', 'love', 'awesome', 'cool', 'good',
    '00s', '90s', '80s', '70s', '60s', '2000s', '2010s', '2020s', '2024', '2025'
}

def obtener_genero(nombre_track, nombre_artista):
    """
    Devuelve el género principal del track (el tag más popular que sea un género real).
    Devuelve 'UNKNOWN' si no encuentra nada.
    """
    datos = llamar_api({
        'method'     : 'track.getTopTags',
        'track'      : nombre_track,
        'artist'     : nombre_artista,
        'autocorrect': 1  # Last.fm corrige errores de escritura
    })
    
    if not datos:
        return 'UNKNOWN'
    
    tags = datos.get('toptags', {}).get('tag', [])
    
    for tag in tags:
        nombre_tag = tag.get('name', '').lower().strip()
        if nombre_tag and nombre_tag not in TAGS_A_IGNORAR:
            return nombre_tag  # primer tag válido = género principal
    
    return 'UNKNOWN'

# Prueba rápida (descomenta para probar):
# print(obtener_genero('Espresso', 'Sabrina Carpenter'))
print('✅ Función lista')

In [ ]:
# --- Ejecutar la recolección ---
# Si ya tienes el CSV guardado, lo carga directamente sin volver a pedir a la API.
# Cambia CSV_PATH si guardas el archivo en otro sitio.

PAISES = [
    'spain', 'united states', 'united kingdom', 'brazil',
    'germany', 'france', 'mexico', 'peru', 'japan', 'chile'
]

TAGS_MUSICA = [
    'pop', 'rock', 'hip-hop', 'electronic', 'latin',
    'indie', 'r&b', 'reggaeton', 'alternative', 'metal',
    'jazz', 'folk', 'punk', 'ambient', 'classic rock'
]

CSV_PATH = 'lastfm_dataset.csv'

if os.path.exists(CSV_PATH):
    # ✅ Ya tenemos datos guardados: cargamos y listo
    print(f'Cargando CSV existente: {CSV_PATH}')
    df_raw = pd.read_csv(CSV_PATH, low_memory=False)
    print(f'✅ {len(df_raw):,} filas cargadas')
else:
    # Primera vez: recoger datos de la API
    print('Iniciando recolección (puede tardar 20-40 min)...')
    todos = []
    
    print('\n[1/3] Top tracks globales')
    todos += recoger_top_tracks_global(n_paginas=150)
    
    print('\n[2/3] Top tracks por país')
    todos += recoger_top_tracks_por_pais(PAISES, paginas_por_pais=10)
    
    print('\n[3/3] Top tracks por género')
    todos += recoger_top_tracks_por_genero(TAGS_MUSICA, paginas_por_tag=10)
    
    df_raw = pd.DataFrame(todos)
    df_raw.to_csv(CSV_PATH, index=False)
    print(f'\n✅ {len(df_raw):,} filas guardadas en {CSV_PATH}')

print(f'\nEstructura del dataset:')
print(f'  Filas: {len(df_raw):,} | Columnas: {df_raw.shape[1]}')
df_raw.head(3)

In [ ]:
# --- Enriquecer géneros para tracks que tienen genre_tag = UNKNOWN ---
# Solo necesario para los tracks de chart.getTopTracks y geo.getTopTracks.
# Los de tag.getTopTracks ya tienen el género asignado.
#
# AVISO: esto hace una petición por cada track único → puede tardar.
# Guarda el resultado en CSV para no repetirlo.

CSV_ENRIQUECIDO = 'lastfm_con_generos.csv'

if os.path.exists(CSV_ENRIQUECIDO):
    print(f'Cargando géneros ya enriquecidos...')
    df_raw = pd.read_csv(CSV_ENRIQUECIDO, low_memory=False)
    n_con_genero = (df_raw['genre_tag'] != 'UNKNOWN').sum()
    print(f'✅ {n_con_genero:,} tracks con género ({n_con_genero/len(df_raw)*100:.0f}%)')
else:
    df_temp = df_raw.copy()
    
    # Tracks únicos que aún no tienen género
    tracks_sin_genero = (
        df_temp[df_temp['genre_tag'] == 'UNKNOWN'][['name', 'artist']]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    print(f'Tracks sin género: {len(tracks_sin_genero):,}')
    print('Enriqueciendo (puede tardar 20-40 min)...')
    
    # Diccionario cache: (nombre, artista) → género
    cache_generos = {}
    
    for i, fila in tracks_sin_genero.iterrows():
        genero = obtener_genero(fila['name'], fila['artist'])
        cache_generos[(fila['name'], fila['artist'])] = genero
        time.sleep(DELAY)
        
        if (i + 1) % 500 == 0:
            n_ok = sum(1 for v in cache_generos.values() if v != 'UNKNOWN')
            print(f'  {i+1:,}/{len(tracks_sin_genero):,} | con género: {n_ok:,}')
            # Guardado parcial por si se interrumpe
            df_temp.to_csv(CSV_ENRIQUECIDO + '.bak', index=False)
    
    # Aplicar los géneros al DataFrame
    mascara = df_temp['genre_tag'] == 'UNKNOWN'
    df_temp.loc[mascara, 'genre_tag'] = df_temp[mascara].apply(
        lambda r: cache_generos.get((r['name'], r['artist']), 'UNKNOWN'), axis=1
    )
    
    df_temp.to_csv(CSV_ENRIQUECIDO, index=False)
    df_raw = df_temp
    print(f'\n✅ Géneros guardados en {CSV_ENRIQUECIDO}')

In [ ]:
# Vistazo rápido al dataset
print('--- Info básica ---')
df_raw.info()

print('\n--- Distribución de country ---')
print(df_raw['country'].value_counts().head(15))

print('\n--- Distribución de genre_tag ---')
print(df_raw['genre_tag'].value_counts().head(15))

---
## 3. Limpieza <a id='3'></a>

**Sobre los datos que tenemos:**
- `GLOBAL` = tracks del ranking global (chart.getTopTracks). **Son los más populares, no los elimines.**
- `UNKNOWN` = tracks de tag.getTopTracks (sin país asignado). **También son útiles.**
- Países reales = tracks de geo.getTopTracks.

**Error frecuente:** eliminar GLOBAL y UNKNOWN elimina el 67% de los datos más valiosos.  
La solución: mantener todos y crear subconjuntos según el análisis.

**Sobre la duración:** Last.fm devuelve la duración en **segundos** (no minutos ni milisegundos).  
Verificación: `Espresso` → 175 en la API → 175/60 = 2.9 min ✅

In [ ]:
df = df_raw.copy()

# 1. Tipos correctos
for col in ['playcount', 'listeners', 'duration']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['name', 'artist', 'country', 'genre_tag']:
    df[col] = df[col].astype(str).str.strip()

# 2. duration = 0 → es un dato faltante (Last.fm devuelve 0 cuando no tiene el dato)
n_ceros = (df['duration'] == 0).sum()
df['duration'] = df['duration'].replace(0, np.nan)
df['duration'] = df['duration'].fillna(df['duration'].median())
print(f'duration=0 convertidos a NaN e imputados: {n_ceros:,}')

# 3. Limpiar strings vacíos
df['genre_tag'] = df['genre_tag'].replace(['', 'nan', 'None'], 'UNKNOWN')
df['country']   = df['country'].replace(['', 'nan', 'None'], 'UNKNOWN')

# 4. Eliminar tracks sin nombre ni artista (sin identidad no sirven)
df = df.dropna(subset=['name', 'artist'])

# 5. Deduplicar: si el mismo track aparece en varios endpoints, nos quedamos con el mayor playcount
antes = len(df)
df = (
    df.sort_values('playcount', ascending=False)
    .drop_duplicates(subset=['name', 'artist'], keep='first')
    .reset_index(drop=True)
)
print(f'Deduplicación: {antes:,} → {len(df):,} tracks únicos ({antes-len(df):,} duplicados eliminados)')

print(f'\n✅ Dataset limpio: {len(df):,} tracks únicos')

In [ ]:
# --- Feature Engineering ---
# Creamos variables derivadas útiles para el EDA y el ML.

# Duración en minutos (la API devuelve segundos)
df['duration_min'] = df['duration'] / 60

# ¿Es una canción corta? (<2.5 min = formato TikTok/Reels)
df['is_short_track'] = (df['duration_min'] < 2.5).astype(int)

# Engagement: cuántas veces escucha el track cada oyente único
# Ratio alto → canción que engancha y se repite
df['engagement'] = df['playcount'] / df['listeners'].replace(0, np.nan)

# Transformación logarítmica (playcount tiene distribución muy sesgada)
# log1p(x) = log(1+x), funciona aunque x sea 0
df['playcount_log']  = np.log1p(df['playcount'])
df['listeners_log']  = np.log1p(df['listeners'])

# Ranking global por popularidad
df['rank_global'] = df['playcount'].rank(ascending=False, method='min').astype(int)

# Estadísticas del artista (agrupadas desde el dataset)
stats_artista = df.groupby('artist').agg(
    n_tracks_artista=('name', 'count'),
    plays_totales_artista=('playcount', 'sum')
).reset_index()

df = df.merge(stats_artista, on='artist', how='left')

# Peso de este track en el catálogo del artista
df['peso_en_artista'] = df['playcount'] / df['plays_totales_artista'].replace(0, np.nan)

# Guardamos df_clean para el notebook de ML
df_clean = df.copy()
df_clean.to_csv('lastfm_clean.csv', index=False)

print(f'✅ df_clean listo')
print(f'   {len(df_clean):,} filas | {df_clean.shape[1]} columnas')
print(f'   Columnas: {list(df_clean.columns)}')
df_clean.head(3)

In [ ]:
# --- ¿8.000 filas son suficientes para ML? ---
# Respuesta: DEPENDE.
#
# Para modelos simples como Random Forest o Logistic Regression:
#   → Con 8.000 filas es suficiente para obtener resultados decentes.
#   → No esperes un modelo perfecto, pero sí uno funcional y explicable.
#
# Para tener más datos:
#   → Aumenta n_paginas en recoger_top_tracks_global() (hasta ~300)
#   → Añade más países en PAISES
#   → Añade más géneros en TAGS_MUSICA
#
# Mínimo recomendado para este proyecto: 5.000 filas con columnas bien rellenas

total = len(df_clean)
con_genero = (df_clean['genre_tag'] != 'UNKNOWN').sum()
con_pais   = (~df_clean['country'].isin(['UNKNOWN', 'GLOBAL'])).sum()

print(f'Total tracks:            {total:,}')
print(f'Con género asignado:     {con_genero:,} ({con_genero/total*100:.0f}%)')
print(f'Con país asignado:       {con_pais:,} ({con_pais/total*100:.0f}%)')
print(f'Con playcount > 0:       {(df_clean["playcount"] > 0).sum():,}')
print()
if total >= 5000:
    print('✅ Volumen suficiente para ML básico')
else:
    print('⚠️  Pocos datos. Aumenta las páginas en la recolección.')

---
## 4. EDA — Exploración visual <a id='4'></a>

In [ ]:
# --- Distribución de popularidad ---
# La escala lineal muestra el sesgo. La log muestra la distribución real.

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

p99 = df_clean['playcount'].quantile(0.99)  # recortamos el 1% extremo

axes[0].hist(df_clean['playcount'].clip(upper=p99), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución Playcount (escala lineal)')
axes[0].set_xlabel('Reproducciones')

axes[1].hist(df_clean['playcount_log'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribución log(Playcount) ← mejor para ML')
axes[1].set_xlabel('log(1 + playcount)')

plt.suptitle('¿Por qué usamos transformación logarítmica?', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Skewness lineal: {df_clean["playcount"].skew():.1f}')
print(f'Skewness log:    {df_clean["playcount_log"].skew():.2f}  ← más cercano a 0 = más normal')

In [ ]:
# --- Top 15 canciones más populares ---

top15 = (
    df_clean
    .sort_values('playcount', ascending=False)
    .head(15)
    [['name', 'artist', 'playcount', 'engagement', 'genre_tag']]
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    top15['name'] + ' — ' + top15['artist'],
    top15['playcount'] / 1e6,
    color=sns.color_palette('Blues_r', 15)
)
ax.set_xlabel('Reproducciones (millones)')
ax.set_title('🏆 Top 15 Canciones por Playcount')
ax.invert_yaxis()

for bar, val in zip(bars, top15['playcount']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.0f}M', va='center', fontsize=9)

plt.tight_layout()
plt.show()

top1 = top15.iloc[0]
print(f'#1: "{top1["name"]}" de {top1["artist"]}')
print(f'   {top1["playcount"]/1e6:.0f}M reproducciones | Engagement: {top1["engagement"]:.1f}x por oyente')

In [ ]:
# --- Top artistas ---

top_artistas = (
    df_clean
    .groupby('artist')
    .agg(
        plays_totales=('playcount', 'sum'),
        n_tracks     =('name', 'count'),
    )
    .sort_values('plays_totales', ascending=False)
    .head(15)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top_artistas['artist'], top_artistas['plays_totales'] / 1e6,
             color=sns.color_palette('Purples_r', 15))
axes[0].invert_yaxis()
axes[0].set_xlabel('Plays totales (millones)')
axes[0].set_title('Top 15 Artistas por Reproducciones')

axes[1].barh(top_artistas['artist'], top_artistas['n_tracks'],
             color=sns.color_palette('Oranges_r', 15))
axes[1].invert_yaxis()
axes[1].set_xlabel('Nº de tracks en el dataset')
axes[1].set_title('Top 15 Artistas por Presencia')

plt.suptitle('🎤 Artistas dominantes', fontweight='bold')
plt.tight_layout()
plt.show()

# Concentración del mercado
top5_share = top_artistas.head(5)['plays_totales'].sum() / df_clean['playcount'].sum() * 100
print(f'Los top 5 artistas concentran el {top5_share:.0f}% de reproducciones totales.')
print('→ Mercado muy concentrado. Para artistas nuevos, el nicho puede ser más efectivo.')

In [ ]:
# --- Análisis por género ---

df_gen = df_clean[df_clean['genre_tag'] != 'UNKNOWN'].copy()

if len(df_gen) == 0:
    print('⚠️  No hay géneros asignados aún. Ejecuta el paso de enriquecimiento (sección 2.4).')
else:
    stats_genero = (
        df_gen
        .groupby('genre_tag')
        .agg(
            n_tracks       =('name', 'count'),
            plays_medio    =('playcount', 'mean'),
            plays_total    =('playcount', 'sum'),
            engagement_medio=('engagement', 'mean'),
            pct_cortas     =('is_short_track', 'mean'),
        )
        .reset_index()
    )
    stats_genero['pct_cortas'] *= 100
    top10 = stats_genero.sort_values('plays_total', ascending=False).head(10)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    palette = sns.color_palette('tab10', 10)

    axes[0,0].bar(top10['genre_tag'], top10['plays_total'] / 1e6, color=palette)
    axes[0,0].set_title('Reproducciones totales')
    axes[0,0].set_ylabel('Millones')
    axes[0,0].tick_params(axis='x', rotation=35)

    axes[0,1].bar(top10['genre_tag'], top10['n_tracks'], color=palette)
    axes[0,1].set_title('Nº de tracks')
    axes[0,1].tick_params(axis='x', rotation=35)

    media_eng = df_clean['engagement'].mean()
    axes[1,0].bar(top10['genre_tag'], top10['engagement_medio'], color=palette)
    axes[1,0].axhline(media_eng, color='red', linestyle='--', alpha=0.6, label='Media global')
    axes[1,0].set_title('Engagement medio (plays/oyente)')
    axes[1,0].legend()
    axes[1,0].tick_params(axis='x', rotation=35)

    axes[1,1].bar(top10['genre_tag'], top10['pct_cortas'], color=palette)
    axes[1,1].set_title('% Canciones cortas (<2.5 min)')
    axes[1,1].set_ylabel('%')
    axes[1,1].tick_params(axis='x', rotation=35)

    plt.suptitle('🎵 Análisis por género', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Insight: el género con más tracks ≠ el más popular.')
    print('Un género con pocos tracks pero alto playcount medio tiene oportunidad de mercado sin saturación.')

In [ ]:
# --- Análisis geográfico ---
# Solo usamos los tracks con país real asignado (los de geo.getTopTracks).
# No eliminamos GLOBAL/UNKNOWN del df_clean — solo los excluimos en este análisis.

df_geo = df_clean[~df_clean['country'].isin(['UNKNOWN', 'GLOBAL'])].copy()

if len(df_geo) == 0:
    print('No hay datos geográficos. Ejecuta geo.getTopTracks en la recolección.')
else:
    stats_pais = (
        df_geo
        .groupby('country')
        .agg(
            n_tracks       =('name', 'count'),
            plays_medio    =('playcount', 'mean'),
            engagement_medio=('engagement', 'mean'),
        )
        .sort_values('plays_medio', ascending=False)
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(stats_pais['country'], stats_pais['plays_medio'] / 1e6,
                 color=sns.color_palette('Blues_r', len(stats_pais)))
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Playcount medio por track (millones)')
    axes[0].set_title('Popularidad media por país')

    top_eng = stats_pais.sort_values('engagement_medio', ascending=False)
    axes[1].barh(top_eng['country'], top_eng['engagement_medio'],
                 color=sns.color_palette('Oranges_r', len(top_eng)))
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Engagement medio')
    axes[1].set_title('Engagement por país')

    plt.suptitle('🌍 Análisis geográfico', fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'País con mayor playcount medio: {stats_pais.iloc[0]["country"]}')

In [ ]:
# --- Heatmap de correlaciones ---
# ¿Qué variables se relacionan más con la popularidad?
# Esto guía qué features usar en el ML.

cols_numericas = [
    'playcount_log', 'listeners_log', 'engagement',
    'duration_min', 'is_short_track',
    'n_tracks_artista', 'peso_en_artista'
]
cols_numericas = [c for c in cols_numericas if c in df_clean.columns]

corr = df_clean[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # solo triángulo inferior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlaciones entre variables', fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

# Variables más correlacionadas con playcount
corr_target = corr['playcount_log'].drop('playcount_log').abs().sort_values(ascending=False)
print('Variables más relacionadas con la popularidad (playcount_log):')
for feat, r in corr_target.head(4).items():
    signo = '+' if corr.loc['playcount_log', feat] > 0 else '-'
    print(f'  {signo} {feat}: r={r:.3f}')

In [ ]:
# --- Duración vs popularidad ---
# ¿Las canciones cortas son más populares?

rangos = pd.cut(
    df_clean['duration_min'],
    bins=[0, 1.5, 2.5, 3.5, 4.5, 6, 100],
    labels=['<1.5m', '1.5-2.5m', '2.5-3.5m', '3.5-4.5m', '4.5-6m', '>6m']
)
pop_por_rango = df_clean.groupby(rangos, observed=True)['playcount_log'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pop_por_rango.plot.bar(ax=axes[0], color=sns.color_palette('RdYlGn', 6))
axes[0].set_title('Popularidad media por duración')
axes[0].set_ylabel('log(Playcount) medio')
axes[0].tick_params(axis='x', rotation=30)

df_clean.boxplot(column='playcount_log', by='is_short_track', ax=axes[1])
axes[1].set_title('Corta (<2.5m) vs Larga')
axes[1].set_xlabel('is_short_track (0=No, 1=Sí)')
axes[1].set_ylabel('log(Playcount)')
plt.suptitle('')

plt.suptitle('⏱️ ¿Afecta la duración a la popularidad?', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

corta = df_clean[df_clean['is_short_track'] == 1]['playcount_log'].mean()
larga = df_clean[df_clean['is_short_track'] == 0]['playcount_log'].mean()
ganador = 'cortas (<2.5 min)' if corta > larga else 'largas (≥2.5 min)'
print(f'Las canciones {ganador} tienen mayor popularidad media.')
print(f'  Cortas: {corta:.3f} | Largas: {larga:.3f}')

---
## 5. ML — Predicción de hits <a id='5'></a>

**¿Qué predecimos?**
- **Clasificación:** ¿es esta canción un hit? (playcount ≥ percentil 75)
- **Regresión:** ¿qué playcount tendrá? (predecimos el valor continuo)

**Variables disponibles para predecir:**

| Variable | Tipo | Valor para ML |
|---|---|---|
| `listeners_log` | numérica | ⭐⭐⭐ Mejor predictor |
| `genre_tag` | categórica | ⭐⭐ El género importa mucho |
| `country` | categórica | ⭐⭐ Diferencias culturales reales |
| `duration_min` | numérica | ⭐ La duración tiene algo de señal |
| `is_short_track` | binaria | ⭐ Formato TikTok |
| `n_tracks_artista` | numérica | ⭐ Artistas con más tracks = más conocidos |
| `peso_en_artista` | numérica | ⭐ ¿Es el hit del artista? |

**Limitación importante:** Last.fm **no tiene** danceability, energy, valence, BPM.  
Para esas features necesitarías Spotify API. Con lo que tenemos, el modelo predice bien la popularidad general pero no las características acústicas de un hit.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, r2_score, mean_absolute_error


In [ ]:
# --- Preparar datos para ML ---

# Target: is_hit (1 = hit, 0 = no hit)
# Umbral: percentil 75 de playcount
umbral_hit = df_clean['playcount'].quantile(0.75)
df_clean['is_hit'] = (df_clean['playcount'] >= umbral_hit).astype(int)

print(f'Umbral de hit: {umbral_hit:,.0f} reproducciones (percentil 75)')
print(f'Hits:     {df_clean["is_hit"].sum():,} ({df_clean["is_hit"].mean()*100:.0f}%)')
print(f'No hits:  {(1-df_clean["is_hit"]).sum():,} ({(1-df_clean["is_hit"].mean())*100:.0f}%)')

# Encodear variables categóricas (texto → número)
df_ml = df_clean.copy()
le_genero = LabelEncoder()
le_pais   = LabelEncoder()

df_ml['genre_encoded']   = le_genero.fit_transform(df_ml['genre_tag'].fillna('UNKNOWN'))
df_ml['country_encoded'] = le_pais.fit_transform(df_ml['country'].fillna('UNKNOWN'))

# Features: qué columnas usamos para predecir
FEATURES = [
    'listeners_log',
    'duration_min',
    'is_short_track',
    'genre_encoded',
    'country_encoded',
    'n_tracks_artista',
    'peso_en_artista',
    'engagement',
]
FEATURES = [f for f in FEATURES if f in df_ml.columns]

# Eliminar filas con NaN en las features
df_ml_ok = df_ml[FEATURES + ['playcount_log', 'is_hit']].dropna()
print(f'\nDataset para ML: {len(df_ml_ok):,} filas | {len(FEATURES)} features')
print(f'Features: {FEATURES}')

In [ ]:
# --- Entrenar modelos ---

X = df_ml_ok[FEATURES]
y_clf = df_ml_ok['is_hit']        # para clasificación
y_reg = df_ml_ok['playcount_log'] # para regresión

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
_, _, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

# Clasificación
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_pred_clf = rf_clf.predict(X_test)

print('\n--- Clasificación (¿es un hit?) ---')
print(classification_report(y_clf_test, y_pred_clf, target_names=['No hit', 'Hit']))

# Regresión
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred_reg = rf_reg.predict(X_test)

print(f'--- Regresión (predecir log de plays) ---')
print(f'R²:  {r2_score(y_reg_test, y_pred_reg):.3f}  (1.0 = perfecto)')
print(f'MAE: {mean_absolute_error(y_reg_test, y_pred_reg):.3f}')

In [ ]:
# --- Feature Importance: ¿qué determina si una canción es un hit? ---

importancias = pd.Series(rf_clf.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
importancias.plot.bar(ax=ax, color=sns.color_palette('Blues_r', len(importancias)))
ax.set_title('🎯 ¿Qué determina si una canción es un hit?', fontweight='bold')
ax.set_ylabel('Importancia relativa')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('Las 3 variables más importantes:')
for feat, val in importancias.head(3).items():
    print(f'  → {feat}: {val*100:.1f}%')

---
## 6. Función de predicción final <a id='6'></a>

Esta función es el **output de negocio** del módulo:  
> "Tu canción tiene X% de probabilidad de ser un hit"

**Para usar en Streamlit:** esta función puede llamarse directamente desde la app.

In [ ]:
def predecir_hit(nombre, artista, duracion_min, genero, pais, oyentes_estimados):
    """
    Predice la probabilidad de que una canción sea un hit.
    
    Parámetros:
        nombre            — nombre de la canción
        artista           — nombre del artista  
        duracion_min      — duración en minutos (ej: 3.5)
        genero            — género musical (ej: 'pop', 'rock')
        pais              — país de lanzamiento (ej: 'spain')
        oyentes_estimados — estimación inicial de oyentes únicos
    
    Devuelve: probabilidad de hit (0-100%)
    """
    # Encodear género y país (si no están en el vocabulario del encoder, usamos 0)
    if genero in le_genero.classes_:
        genre_enc = le_genero.transform([genero])[0]
    else:
        genre_enc = 0

    if pais in le_pais.classes_:
        country_enc = le_pais.transform([pais])[0]
    else:
        country_enc = 0

    # Crear el vector de features
    datos = pd.DataFrame([{
        'listeners_log'  : np.log1p(oyentes_estimados),
        'duration_min'   : duracion_min,
        'is_short_track' : int(duracion_min < 2.5),
        'genre_encoded'  : genre_enc,
        'country_encoded': country_enc,
        'n_tracks_artista': 1,    # artista nuevo = 1 track
        'peso_en_artista': 1.0,   # único track del artista = 100%
        'engagement'     : 5.0,   # engagement inicial estimado
    }])
    datos = datos[FEATURES]  # mismo orden que el modelo espera

    probabilidad = rf_clf.predict_proba(datos)[0][1] * 100

    if probabilidad >= 70:
        clasificacion = '🚀 Hit potencial'
    elif probabilidad >= 45:
        clasificacion = '🟡 Potencial medio'
    else:
        clasificacion = '📉 Bajo potencial'

    print('='*50)
    print(f'  🎵 {nombre} — {artista}')
    print('='*50)
    print(f'  Probabilidad de hit:  {probabilidad:.1f}%')
    print(f'  Clasificación:        {clasificacion}')
    print(f'  Género:               {genero}')
    print(f'  País:                 {pais}')
    print(f'  Duración:             {duracion_min:.1f} min', end='')
    print(f' (formato corto ⏱️)' if duracion_min < 2.5 else '')
    print('='*50)

    return probabilidad


# --- Ejemplo de uso ---
predecir_hit(
    nombre='Mi Canción',
    artista='Mi Artista',
    duracion_min=2.3,
    genero='pop',
    pais='spain',
    oyentes_estimados=50000
)

---
## 💡 Ideas para Streamlit / App web

Con lo que tienes, puedes construir una app con estas secciones:

### Página 1 — Predictor de hit
```python
# En app.py:
import streamlit as st
import pickle

# Cargar modelo y encoders guardados
rf_clf = pickle.load(open('modelo_hits.pkl', 'rb'))
le_genero = pickle.load(open('le_genero.pkl', 'rb'))

st.title('🎵 ¿Será un hit?')
duracion   = st.slider('Duración (min)', 1.0, 8.0, 3.5)
genero     = st.selectbox('Género', ['pop', 'rock', 'hip-hop', 'electronic'])
oyentes    = st.number_input('Oyentes estimados', value=10000)

if st.button('Predecir'):
    prob = predecir_hit('Mi canción', 'Mi artista', duracion, genero, 'spain', oyentes)
    st.metric('Probabilidad de hit', f'{prob:.0f}%')
    st.progress(prob / 100)
```

### Página 2 — Dashboard del mercado
- Top 15 canciones (barplot)
- Géneros por popularidad (barplot)
- Mapa de calor géneros × países

### Página 3 — Explorador de artistas
- Input: nombre del artista
- Output: sus tracks en el dataset, popularidad media, géneros

### Cómo guardar y cargar el modelo
```python
import pickle

# Guardar (en el notebook)
pickle.dump(rf_clf,    open('models/modelo_hits.pkl', 'wb'))
pickle.dump(le_genero, open('models/le_genero.pkl', 'wb'))
pickle.dump(le_pais,   open('models/le_pais.pkl', 'wb'))

# Cargar (en Streamlit)
rf_clf    = pickle.load(open('models/modelo_hits.pkl', 'rb'))
le_genero = pickle.load(open('models/le_genero.pkl', 'rb'))
```

### Ideas de ML adicionales con las variables actuales

| Idea | Variables necesarias | Factibilidad |
|---|---|---|
| Predecir si un track será hit | listeners, genre, country, duration | ✅ Ya tienes esto |
| Clustering de géneros musicales | playcount, engagement, pct_cortas | ✅ KMeans con tus datos |
| Ranking de países con más potencial | plays_medio por país | ✅ Con geo.getTopTracks |
| Detectar artistas emergentes | n_tracks_artista, plays_medio, tendencia | ✅ Con datos temporales |
| Predecir viralidad por duración | is_short_track + playcount | ✅ Ya tienes esto |
| Audio features (danceability, BPM...) | — | ❌ Necesitas Spotify API |

### Para el backup_tracks.csv (tiene 'published' = fechas reales)
El archivo `backup_tracks.csv` tiene un campo `published` con fechas reales de publicación.  
Esto permite análisis temporales reales: qué géneros triunfan en verano, tendencias por año, etc.

In [ ]:
# --- Guardar modelos para usar en Streamlit ---
import pickle
import os

os.makedirs('models', exist_ok=True)

pickle.dump(rf_clf,    open('models/modelo_hits_clf.pkl', 'wb'))
pickle.dump(rf_reg,    open('models/modelo_plays_reg.pkl', 'wb'))
pickle.dump(le_genero, open('models/le_genero.pkl', 'wb'))
pickle.dump(le_pais,   open('models/le_pais.pkl', 'wb'))

# Guardar también las features usadas (importante: el orden debe coincidir)
with open('models/features.txt', 'w') as f:
    f.write('\n'.join(FEATURES))

print('✅ Modelos guardados en /models/')
print('   Para cargarlos en Streamlit:')
print('   rf_clf = pickle.load(open("models/modelo_hits_clf.pkl", "rb"))')